In [65]:
from veripulse.pulse import PulseConfig, PulseResult, choi_optimise_secret_indep, nvcenter_system
import re
import json
import numpy as np
from pathlib import Path
from typing import Optional
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter1d
from veripulse.sdp import calc_secret_indep
from veripulse.gates import rx
from qutip import (
    Qobj, basis, ket2dm, operator_to_vector, vector_to_operator, fidelity
)
from veripulse.gates import pack_subspace_states, extract_subspace_states



%matplotlib inline

In [72]:
def evolve_liouvillian(amps_2d, cfg, L_drift, L_ctrl, rho_vec_init):
    """
    Evolve vectorised state under piecewise-constant Liouvillian control.

    Parameters
    ----------
    amps_2d : ndarray, shape (num_tslots, num_ctrls)
    cfg : PulseConfig
    L_drift : Qobj (superoperator)
    L_ctrl : list of Qobj (superoperators)
    rho_vec_init : Qobj (vectorised density matrix)

    Returns
    -------
    rho_vec_final : Qobj (vectorised density matrix)
    """
    dt = cfg.evo_time / cfg.num_tslots
    rho_vec = rho_vec_init

    for k in range(amps_2d.shape[0]):
        L_total = L_drift
        for c, Lc in enumerate(L_ctrl):
            L_total = L_total + amps_2d[k, c] * Lc
        prop = (L_total * dt).expm()
        rho_vec = prop * rho_vec

    return rho_vec


def get_pulse_for_state(result, state_idx):
    """
    Extract (num_tslots, 2) amp array for a given state index.
    """
    amps = result.final_amps
    if amps.ndim == 3:
        return amps[state_idx]
    else:
        return amps[:, state_idx * 2 : state_idx * 2 + 2]


def find_state_idx(state_labels, angle):
    """Find the index of the closest state_label to a given angle (mod 2pi)."""
    angle_mod = angle % (2 * np.pi)
    labels_mod = np.array(state_labels) % (2 * np.pi)
    diffs = np.abs(labels_mod - angle_mod)
    diffs = np.minimum(diffs, 2 * np.pi - diffs)
    return int(np.argmin(diffs))


def randomised_si(result, cfg, rho_init, unitaries, n_split=2, n_random=10, rng=None):
    """
    Compute SI with randomised pulse composition.

    Parameters
    ----------
    result : PulseResult
    cfg : PulseConfig
    rho_init : Qobj (single-qubit density matrix)
    unitaries : list of Qobj (target unitaries per state_label)
    n_split : int
    n_random : int
    rng : np.random.Generator, optional

    Returns
    -------
    si_mean, si_std, si_list
    """
    if rng is None:
        rng = np.random.default_rng()

    state_labels = result.state_labels
    K = len(state_labels)
    d = 2
    is_avg = result.final_amps.ndim == 2

    if cfg.num_tslots != result.config.num_tslots:
        raise ValueError(
            f"cfg.num_tslots={cfg.num_tslots} does not match "
            f"result.config.num_tslots={result.config.num_tslots}"
        )
    if not np.isclose(cfg.evo_time, result.config.evo_time):
        raise ValueError(
            f"cfg.evo_time={cfg.evo_time} does not match "
            f"result.config.evo_time={result.config.evo_time}"
        )

    rho_targs = [U * rho_init * U.dag() for U in unitaries]

    si_list = []

    for trial in range(n_random):
        if is_avg and n_split == 1:
            # GRAPE_AVG: use pack_subspace_states + evolve in K*d space
            rotations = [rx(theta) for theta in state_labels]
            vRho_init, vRho_target, U_big = pack_subspace_states(rotations, rho_init)

            L_drift, L_ctrl = nvcenter_system(K, cfg)

            # Rebuild rho_init_big with correct tensor dims for nvcenter_system
            rho_init_big = Qobj(
                np.kron(np.eye(K), rho_init.full()) / K,
                dims=[[K, d], [K, d]]
            )
            vRho_init = operator_to_vector(rho_init_big)

            # Reorder amps from interleaved [x0,y0,x1,y1,...] 
            # to grouped [x0,x1,...,x7,y0,y1,...,y7] to match L_ctrl
            amps = result.final_amps
            x_cols = amps[:, 0::2]  # columns 0,2,4,...
            y_cols = amps[:, 1::2]  # columns 1,3,5,...

            rho_vec_final = evolve_liouvillian(
                amps, cfg, L_drift, L_ctrl, vRho_init
            )

            rho_big = vector_to_operator(rho_vec_final) * K
            rho_ests = [Qobj(rho_big[d * i: d * i + d, d * i: d * i + d])
                        for i in range(K)]

        else:
            # Per-state: GRAPE/CRAB, or composition with n_split > 1
            L_drift_1, L_ctrl_1 = nvcenter_system(1, cfg)
            rho_ests = []

            for target_theta in state_labels:
                if n_split == 1:
                    angles = [target_theta]
                else:
                    random_angles = rng.uniform(0, 2 * np.pi, size=n_split - 1)
                    last_angle = (target_theta - np.sum(random_angles)) % (2 * np.pi)
                    angles = list(random_angles) + [last_angle]

                rho_vec = operator_to_vector(rho_init)
                for angle in angles:
                    state_idx = find_state_idx(state_labels, angle)
                    pulse_amps = get_pulse_for_state(result, state_idx)
                    rho_vec = evolve_liouvillian(
                        pulse_amps, cfg, L_drift_1, L_ctrl_1, rho_vec
                    )
                rho_ests.append(vector_to_operator(rho_vec))

        si = choi_optimise_secret_indep(
            [rho.full() for rho in rho_targs],
            [rho.full() for rho in rho_ests],
        )
        si_list.append(si.objective)

        # Debug: print fidelity error for each state
        for j, (rt, re) in enumerate(zip(rho_targs, rho_ests)):
            F = fidelity(rt, re)
            err = 1 - F 
            print(f"  state {j} (θ={state_labels[j]:.3f}): err={err:.6e}  (initial={result.err_ul_list[j]:.6e})")

    return np.mean(si_list), np.std(si_list), si_list

In [75]:
path='/users/home/gustiani/VeriPulse/data/GRAPE_AVG_p40_det0.00_err0.00-lam0.050-5.json'
result = PulseResult.load(path)
cfg = result.config
rho_init = ket2dm(basis(2, 0))
unitaries = [Qobj(rx(theta)) for theta in result.state_labels]

si_mean, si_std, si_list = randomised_si(
    result, cfg, rho_init, unitaries, n_split=3, n_random=1
)
si_list

  state 0 (θ=0.000): err=6.951383e-01  (initial=1.570381e-04)
  state 1 (θ=0.785): err=2.446723e-01  (initial=1.188793e-03)
  state 2 (θ=1.571): err=2.372030e-01  (initial=2.442049e-03)
  state 3 (θ=2.356): err=1.275753e-01  (initial=2.874176e-03)
  state 4 (θ=3.142): err=5.934382e-02  (initial=2.348327e-03)
  state 5 (θ=3.927): err=1.336770e-01  (initial=2.495923e-03)
  state 6 (θ=4.712): err=5.690508e-01  (initial=2.821766e-03)
  state 7 (θ=5.498): err=6.931633e-02  (initial=3.491723e-03)


[np.float64(0.3422872150107681)]

In [51]:
result.si

0.001129142169258838

In [52]:
result.err_ul_list

[0.000157038062940984,
 0.0011887930154418358,
 0.0024420489465807327,
 0.002874176252940397,
 0.0023483271204151057,
 0.002495922791713845,
 0.0028217662514292696,
 0.0034917234119000717]

In [62]:
path2='/users/home/gustiani/VeriPulse/data/GRAPE_p80_det0.00_err0.00-2.json'
result2 = PulseResult.load(path2)
cfg2 = result2.config
rho_init = ket2dm(basis(2, 0))
unitaries = [Qobj(rx(theta)) for theta in result2.state_labels]

si_mean, si_std, si_list = randomised_si(
    result2, cfg2, rho_init, unitaries, n_split=1, n_random=1
)
si_list

  state 0 (θ=0.000): err=1.970999e-04  (expected=1.971000e-04)
  state 1 (θ=0.785): err=5.342811e-04  (expected=5.343370e-04)
  state 2 (θ=1.571): err=2.653670e-03  (expected=2.653659e-03)
  state 3 (θ=2.356): err=2.920213e-03  (expected=2.920201e-03)
  state 4 (θ=3.142): err=2.310381e-03  (expected=2.310381e-03)
  state 5 (θ=3.927): err=2.717765e-03  (expected=2.717993e-03)
  state 6 (θ=4.712): err=2.492721e-03  (expected=2.492721e-03)
  state 7 (θ=5.498): err=1.164481e-03  (expected=1.164531e-03)


[np.float64(0.00374328815481705)]

In [30]:
result2.err_ul_list

[0.0001970999710189103,
 0.0005343369722469182,
 0.002653659147804577,
 0.002920200965874531,
 0.002310380532691614,
 0.002717993015734832,
 0.002492721317894908,
 0.001164531371081079]

In [31]:
result2.si

0.0012548804859046378